# 리포트 48 — 같은 자원격자를 독립 변조기에 넣어 상관 1.0000 을 얻었다

> ### 한 일
> **우리 변조기와 Sionna PHY 의 `OFDMModulator` 에 같은 자원격자를 넣고 두 시간파형의 상관과 NMSE 를 재, 변조 단계를 독립 구현으로 채점했다.**

### 결과
1. 세 파형 모두 상관이 소수 넷째 자리까지 1 이다 — WiFi 1.0000 [^1] · LTE 1.0000 [^2] · 5G 1.0000 [^3].
2. NMSE 는 WiFi -138.3 [^4] · LTE -135.6 [^5] · 5G -135.2 dB [^6] 로 float32 반올림 바닥에 붙는다.
3. 대조의 **분해력**을 같은 표에 싣는다 — 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 LTE 상관이 0.06 [^7], 5G 가 0.05 [^8] 로 무너진다. CP 가 심볼마다 같은 WiFi 는 1.0000 [^9] 로 남는다.
4. 대조는 G3(풀로드) 격자에서 돈다 — 표본 수 61440 [^10] · $f_s$ 122.88 MHz [^11] (5G 기준).

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 무엇을 채점하나 | 격자를 신호로 바꾸는 **변조 단계**다 — IFFT 규약(fftshift 방향·정규화), CP 복사, 심볼별 이어붙이기 순서 |
| 상대 구현 | `sionna.phy.ofdm.OFDMModulator`(Sionna 2.0.1) — 우리 코드를 한 줄도 공유하지 않는다 |
| 분해력 시험 | 재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기는 대조군을 함께 돌려, 대조가 무엇을 잡아낼 수 있는지를 같은 표에 적는다 |
| 자원격자 자체 | 파일럿 좌표·가드밴드·DC 널은 규격서를 읽어 `src/waveforms.py` 에 세웠고, X410 캡처 대조는 실측 캠페인이 한다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json` |
| 소요 | ① 3412 s [^12] · ② CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 44 «상시이면서 내용을 미리 아는 신호는 표준마다…»](44_illuminators.ipynb) | 세 표준의 자원격자와 상시 기준신호 |

---

## 무엇을 채점하는가

격자를 신호로 바꾸는 **변조 단계**를 독립 구현으로 채점한다. 같은 자원격자를 Sionna PHY 의 `sionna.phy.ofdm.OFDMModulator` 에 넣고, 우리 변조기 출력과 상관·NMSE 를 잰다.

| 이 대조가 확인하는 것 | 무엇으로 |
|---|---|
| IFFT 규약 — fftshift 방향 · 정규화 | 두 구현의 시간파형 상관 |
| CP 복사와 심볼별 이어붙이기 순서 | 심볼별 CP 배열을 뺀 대조군과 비교 |
| 두 독립 구현의 시간파형 일치 | NMSE 바닥 |

![report03_f5_crosscheck](../outputs/figures/report03_f5_crosscheck.png)

**그림 1.** 같은 자원격자를 두 변조기에 넣으면 같은 시간파형이 나오는가?

## 채점 결과

| 표준 | 표본 수 | $f_s$ | 상관 | NMSE | CP 앞머리 | CP 배열을 뺀 대조군 |
|---|---|---|---|---|---|---|
| WiFi 802.11ac | 4160 [^13] | 80.00 MHz [^14] | 1.0000 [^1] | -138.3 dB [^4] | `[64]` | 1.0000 [^9] |
| LTE Rel-9 | 30720 [^15] | 30.72 MHz [^16] | 1.0000 [^2] | -135.6 dB [^5] | `[160, 144, 144, 144]` | 0.0634 [^7] |
| 5G NR Rel-16 | 61440 [^10] | 122.88 MHz [^11] | 1.0000 [^3] | -135.2 dB [^6] | `[352, 288, 288, 288]` | 0.0451 [^8] |

## 마지막 열이 대조의 분해력이다

두 구현이 같은 오해를 공유하면 대조가 통과해도 정보가 0 이다. 그 반론에 미리 답하려고 **일부러 틀린 대조군**을 같은 표에 싣는다.

재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 두 번째 심볼부터 시간축이 어긋난다. CP 길이가 심볼마다 다른 LTE·5G 에서 상관이 무너지고, CP 가 심볼마다 같은 WiFi 는 그대로 1 이다 — 대조가 무엇을 잡아내는지가 그 열에 적혀 있다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| X410 으로 실제 셀을 캡처해 `src/waveforms.py` 의 격자와 대조한다 | CRS · SSB · VHT-LTF 의 격자 좌표가 실측으로 확정된다 | [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |
| 복조·등화 단계까지 같은 방식으로 채점한다 | 변조 밖 사슬의 독립 대조가 어디까지 서는지가 확정된다 | `src/waveforms_sionna.py` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.corr` | 1 |
| [^2] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.corr` | 1 |
| [^3] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.corr` | 1 |
| [^4] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.nmse_db` | -138.3 |
| [^5] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.nmse_db` | -135.6 |
| [^6] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.nmse_db` | -135.2 |
| [^7] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.corr_bug` | 0.06338 |
| [^8] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.corr_bug` | 0.04507 |
| [^9] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.corr_bug` | 1 |
| [^10] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.n` | 61440 |
| [^11] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.fs_mhz` | 122.9 |
| [^12] | `outputs/report2_waveform_rcs.json` | `meta.runtime_s` | 3412 |
| [^13] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.n` | 4160 |
| [^14] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.fs_mhz` | 80 |
| [^15] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.n` | 30720 |
| [^16] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.fs_mhz` | 30.72 |